# 02 — Sensitivity Analysis & Pareto Front (Phase 1 LHS Screening)

**Mục tiêu:**
1. Tính **Standardized Regression Coefficients (SRC)** cho Poisson ratio $\nu_{12}$ và các output khác.
2. Ước lượng **Sobol indices** thông qua Gaussian Process surrogate.
3. Kiểm định **ANOVA** một chiều cho từng tham số.
4. **Phân loại tham số** thành: *nhạy mạnh*, *nhạy cục bộ*, *không nhạy*.
5. **Vẽ Pareto front** giữa auxetic behavior (giá trị mục tiêu) và chi phí tính toán.

> **Lưu ý:** Phase 1 chỉ chạy objective `auxetic` nên không có dữ liệu tần số thực tế.
> Các phân tích dưới đây được thực hiện trên các output có sẵn: $\nu_{12}$, $\nu_{21}$, `obj_value`, `n_iters`.

In [ ]:
# ── 1. Imports & cấu hình ──
import json, os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
from scipy.stats import f_oneway
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel, WhiteKernel

warnings.filterwarnings('ignore', category=FutureWarning)

PHASE1_DIR = Path('../outputs/pipeline/phase1')
OUTPUT_DIR = Path('../outputs/figures/lhs_screening')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Tham số đầu vào (6 biến thiết kế)
PARAM_NAMES = ['volfrac', 'penal', 'rmin', 'move', 'void_size_frac', 'rotation_deg']
PARAM_LABELS = {
    'volfrac': 'Volfrac',
    'penal': 'Penal',
    'rmin': 'Rmin',
    'move': 'Move',
    'void_size_frac': 'Void size frac.',
    'rotation_deg': 'Rotation (deg)',
}

plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'figure.facecolor': 'white',
    'savefig.bbox': 'tight',
})

print('Cấu hình hoàn tất.')

---
## 1. Load & kết hợp dữ liệu từ tất cả seed

In [ ]:
# ── Load tất cả CSV từ mỗi seed ──
seed_dirs = sorted([d for d in PHASE1_DIR.iterdir() if d.is_dir() and not d.name.startswith('_')])
print(f'Tìm thấy {len(seed_dirs)} seeds:')

frames = []
for sd in seed_dirs:
    csv_path = sd / 'auxetic' / f'phase1_{sd.name}_auxetic.csv'
    if csv_path.exists():
        df_seed = pd.read_csv(csv_path)
        df_seed['seed'] = sd.name
        frames.append(df_seed)
        print(f'  ✓ {sd.name}: {len(df_seed)} samples')
    else:
        print(f'  ✗ {sd.name}: CSV not found')

df = pd.concat(frames, ignore_index=True)
n_total = len(df)
n_success = df['success'].sum()
n_converged = df['converged'].sum()
print(f'\nTổng: {n_total} mẫu ({n_success} thành công, {n_converged} hội tụ)')
print(f'Cột: {list(df.columns)}')

In [ ]:
# ── Thống kê mô tả nhanh ──
print('Thống kê các biến đầu ra:')
desc = df[['v12', 'v21', 'obj_value', 'n_iters', 'elapsed_time']].describe().round(4)
desc

In [ ]:
# ── Histogram các output ──
fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

titles = ['Poisson ratio $\\nu_{12}$', 'Poisson ratio $\\nu_{21}$', 'Objective value']
columns = ['v12', 'v21', 'obj_value']
colors = ['#3b82f6', '#8b5cf6', '#ef4444']

for ax, col, title, color in zip(axes, columns, titles, colors):
    ax.hist(df[col].dropna(), bins=20, color=color, edgecolor='white', alpha=0.8)
    ax.set_xlabel(title)
    ax.set_ylabel('Count')
    ax.axvline(df[col].mean(), color='black', ls='--', lw=1, label=f'Mean={df[col].mean():.3f}')
    ax.legend(fontsize=8)

fig.suptitle('Phân phối các biến đầu ra Phase 1', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'output_distributions.png')
plt.show()

---
## 2. Standardized Regression Coefficients (SRC)

Hồi quy tuyến tính trên dữ liệu đã chuẩn hóa $Z$-score:

$$
y_{\text{std}} = \sum_j \beta_j \, x_{j,\text{std}} + \varepsilon
$$

Hệ số $\beta_j$ chính là **SRC** — cho biết mức độ ảnh hưởng của tham số $j$ lên đầu ra.

In [ ]:
def compute_src(X: np.ndarray, y: np.ndarray, param_names: list) -> dict:
    """Tính SRC (Standardized Regression Coefficients)."""
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    ys = StandardScaler().fit_transform(y.reshape(-1, 1)).ravel()
    
    reg = LinearRegression()
    reg.fit(Xs, ys)
    
    n, p = Xs.shape
    r2 = reg.score(Xs, ys)
    r2_adj = 1.0 - (1.0 - r2) * (n - 1) / (n - p - 1) if n > p + 1 else r2
    
    # p-value cho mỗi hệ số (F-test tương đương t-test bậc 2)
    y_pred = reg.predict(Xs)
    mse = np.sum((ys - y_pred)**2) / (n - p - 1)
    var_b = mse * np.linalg.inv(Xs.T @ Xs).diagonal()
    t_stat = reg.coef_ / np.sqrt(np.maximum(var_b, 1e-15))
    p_values = 2 * (1 - stats.t.cdf(np.abs(t_stat), n - p - 1))
    
    return {
        'coef': dict(zip(param_names, reg.coef_)),
        'p_value': dict(zip(param_names, p_values)),
        'r2': r2,
        'r2_adj': r2_adj,
    }


# ── Chọn dữ liệu hợp lệ ──
valid = df['success'] & df['converged'] & df[PARAM_NAMES].notna().all(axis=1)
X = df.loc[valid, PARAM_NAMES].values.astype(float)
print(f'Số mẫu hợp lệ cho SRC: {len(X)}')

outputs = ['v12', 'v21', 'obj_value', 'n_iters']
src_results = {}

fig, axes = plt.subplots(1, len(outputs), figsize=(14, 4))

for idx, out in enumerate(outputs):
    y = df.loc[valid, out].values.astype(float)
    src = compute_src(X, y, PARAM_NAMES)
    src_results[out] = src
    
    coeffs = np.array([src['coef'][p] for p in PARAM_NAMES])
    colors = ['#22c55e' if c > 0 else '#ef4444' for c in coeffs]
    
    ax = axes[idx]
    bars = ax.barh(range(len(PARAM_NAMES)), coeffs, color=colors, edgecolor='white', height=0.6)
    ax.set_yticks(range(len(PARAM_NAMES)))
    ax.set_yticklabels([PARAM_LABELS[p] for p in PARAM_NAMES])
    ax.axvline(0, color='black', lw=0.8)
    ax.set_xlabel('SRC ($\\beta$)')
    ax.set_title(f'{out}\n$R^2={src["r2"]:.3f}$', fontsize=11)
    ax.grid(axis='x', alpha=0.3)
    
    # Thêm dấu * cho p < 0.05
    for i, p in enumerate(PARAM_NAMES):
        if src['p_value'][p] < 0.05:
            ax.text(coeffs[i] + 0.015 * (1 if coeffs[i] >= 0 else -1), i, '*',
                    va='center', fontsize=14, color='black')

fig.suptitle('Standardized Regression Coefficients (SRC) — dấu *: p<0.05',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'src_coefficients.png')
plt.show()

# In chi tiết
print('\n' + '='*70)
print('  SRC DETAILS')
print('='*70)
for out in outputs:
    src = src_results[out]
    print(f'\n  ► {out}  (R²={src["r2"]:.4f}, R²_adj={src["r2_adj"]:.4f})')
    for p in PARAM_NAMES:
        sig = '*' if src['p_value'][p] < 0.05 else ''
        print(f'      {p:20s}  β = {src["coef"][p]:+8.4f}  p = {src["p_value"][p]:.4f} {sig}')

---
## 3. Sobol Indices (ước lượng qua GP surrogate)

Vì Phase 1 là LHS screening (không có replicated sampling cho Sobol chuẩn), ta dùng
**Gaussian Process Regression** làm surrogate model, sau đó tính Sobol indices
bằng phân tích phương sai trên surrogate.

Công thức ước lượng Saltelli (2010):
- $S_{1j} = \frac{V[E(Y|X_j)]}{V[Y]}$ — ảnh hưởng chính (first-order)
- $S_{Tj} = 1 - \frac{V[E(Y|X_{\sim j})]}{V[Y]}$ — tổng ảnh hưởng (total-order)

In [ ]:
def compute_sobol_gp_saltelli(
    X: np.ndarray,
    y: np.ndarray,
    param_names: list,
    n_mc: int = 5000,
) -> dict:
    """
    Tính Sobol indices thông qua GP surrogate + ước lượng Saltelli.

    Args:
        X: Ma trận thiết kế (N, d).
        y: Vector output (N,).
        param_names: Tên tham số.
        n_mc: Số mẫu MC cho ước lượng tích phân.

    Returns:
        Dict với 'S1', 'ST', 'S1_conf', 'ST_conf'.
    """
    n, d = X.shape
    
    # Chuẩn hóa X về [0, 1] cho GP
    X_min, X_max = X.min(axis=0), X.max(axis=0)
    X_norm = (X - X_min) / (X_max - X_min + 1e-15)
    y_mean, y_std = y.mean(), y.std()
    y_norm = (y - y_mean) / (y_std + 1e-15)
    
    # GP với kernel Matern 5/2 + noise
    from sklearn.gaussian_process.kernels import Matern
    kernel = ConstantKernel(1.0) * Matern(length_scale=np.ones(d), nu=2.5) + WhiteKernel(0.1)
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True, n_restarts_optimizer=3, random_state=42)
    gp.fit(X_norm, y_norm)
    print(f'    GP kernel: {gp.kernel_}')
    
    # Saltelli sampling
    rng = np.random.RandomState(42)
    A = rng.uniform(0, 1, size=(n_mc, d))
    B = rng.uniform(0, 1, size=(n_mc, d))
    
    # Dự đoán trên A, B
    y_A = gp.predict(A).ravel()
    y_B = gp.predict(B).ravel()
    f0 = y_A.mean()
    V = y_A.var()
    
    S1, ST = {}, {}
    for j in range(d):
        # Ma trận AB_j và BA_j
        AB_j = A.copy()
        AB_j[:, j] = B[:, j]
        BA_j = B.copy()
        BA_j[:, j] = A[:, j]
        
        y_AB = gp.predict(AB_j).ravel()
        y_BA = gp.predict(BA_j).ravel()
        
        S1[param_names[j]] = (np.mean(y_B * (y_AB - y_A))) / (V + 1e-15)
        ST[param_names[j]] = (np.mean((y_A - y_AB)**2)) / (2 * V + 1e-15)
    
    return {'S1': S1, 'ST': ST, 'V': float(V), 'f0': float(f0)}


# ── Chạy Sobol cho v12 và obj_value ──
print('Tính Sobol indices qua GP surrogate...')
sobol_targets = ['v12', 'obj_value']
sobol_results = {}

for target in sobol_targets:
    print(f'\n  ► {target}')
    y = df.loc[valid, target].values.astype(float)
    sobol_results[target] = compute_sobol_gp_saltelli(X, y, PARAM_NAMES, n_mc=3000)

print('\nHoàn tất tính Sobol indices.')

In [ ]:
# ── Vẽ Sobol indices ──
n_targets = len(sobol_targets)
fig, axes = plt.subplots(1, n_targets, figsize=(7 * n_targets, 4.5))
if n_targets == 1:
    axes = [axes]

for idx, target in enumerate(sobol_targets):
    ax = axes[idx]
    s1 = np.array([sobol_results[target]['S1'][p] for p in PARAM_NAMES])
    st = np.array([sobol_results[target]['ST'][p] for p in PARAM_NAMES])
    
    x = np.arange(len(PARAM_NAMES))
    w = 0.3
    
    ax.bar(x - w/2, s1, w, label='S1 (first-order)', color='#3b82f6', edgecolor='white')
    ax.bar(x + w/2, st, w, label='ST (total)', color='#f59e0b', edgecolor='white')
    
    ax.set_xticks(x)
    ax.set_xticklabels([PARAM_LABELS[p] for p in PARAM_NAMES], rotation=30, ha='right')
    ax.set_ylabel('Sobol index')
    ax.set_title(f'Sobol indices — {target}', fontsize=12)
    ax.legend()
    ax.set_ylim(0, 1)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sobol_indices.png')
plt.show()

# In chi tiết
print('\n' + '='*70)
print('  SOBOL INDICES (GP surrogate)')
print('='*70)
for target in sobol_targets:
    print(f'\n  ► {target}')
    for p in PARAM_NAMES:
        s1 = sobol_results[target]['S1'][p]
        st = sobol_results[target]['ST'][p]
        print(f'      {p:20s}  S1 = {s1:.4f}   ST = {st:.4f}')

---
## 4. ANOVA — Phân tích phương sai một chiều

Chia mỗi tham số thành 3–5 khoảng đều nhau, kiểm định $H_0$: trung bình output
bằng nhau giữa các khoảng. Dùng $F$-statistic và $p$-value.

In [ ]:
def compute_oneway_anova(
    df: pd.DataFrame,
    param: str,
    target: str,
    n_bins: int = 5,
) -> dict:
    """Tính ANOVA một chiều cho một tham số."""
    # Chia thành n_bins khoảng đều nhau
    bins = np.linspace(df[param].min(), df[param].max() + 1e-10, n_bins + 1)
    labels = range(n_bins)
    df['_bin'] = pd.cut(df[param], bins=bins, labels=labels)
    
    groups = [group[target].values for _, group in df.groupby('_bin', observed=True)]
    
    if len(groups) < 2:
        return {'F': 0.0, 'p': 1.0, 'n_groups': len(groups)}
    
    f_stat, p_val = f_oneway(*groups)
    return {'F': f_stat, 'p': p_val, 'n_groups': len(groups)}


# ── Chạy ANOVA cho tất cả param × target ──
anova_targets = ['v12', 'v21', 'obj_value', 'n_iters']
anova_records = []

for param in PARAM_NAMES:
    for target in anova_targets:
        res = compute_oneway_anova(df.loc[valid], param, target, n_bins=5)
        anova_records.append({
            'param': param,
            'target': target,
            'F': res['F'],
            'p': res['p'],
            'significant': res['p'] < 0.05,
        })

df_anova = pd.DataFrame(anova_records)

# ── Heatmap ANOVA ──
pivot_f = df_anova.pivot(index='param', columns='target', values='F')
pivot_p = df_anova.pivot(index='param', columns='target', values='p')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

import matplotlib.colors as mcolors

# F-statistic heatmap
im1 = axes[0].imshow(pivot_f.values, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(len(anova_targets)))
axes[0].set_xticklabels(anova_targets, rotation=30, ha='right')
axes[0].set_yticks(range(len(PARAM_NAMES)))
axes[0].set_yticklabels([PARAM_LABELS[p] for p in PARAM_NAMES])
axes[0].set_title('F-statistic (ANOVA)')
fig.colorbar(im1, ax=axes[0], shrink=0.8)
for i in range(len(PARAM_NAMES)):
    for j in range(len(anova_targets)):
        axes[0].text(j, i, f'{pivot_f.values[i,j]:.1f}', ha='center', va='center',
                     fontsize=9, color='black' if pivot_f.values[i,j] < 10 else 'white')

# p-value heatmap (log scale)
p_log = -np.log10(np.maximum(pivot_p.values, 1e-15))
im2 = axes[1].imshow(p_log, cmap='Blues', aspect='auto')
axes[1].set_xticks(range(len(anova_targets)))
axes[1].set_xticklabels(anova_targets, rotation=30, ha='right')
axes[1].set_yticks(range(len(PARAM_NAMES)))
axes[1].set_yticklabels([PARAM_LABELS[p] for p in PARAM_NAMES])
axes[1].set_title('$-$log$_{10}(p)$ (ANOVA)')
fig.colorbar(im2, ax=axes[1], shrink=0.8)
for i in range(len(PARAM_NAMES)):
    for j in range(len(anova_targets)):
        pv = pivot_p.values[i, j]
        label = f'{pv:.3f}' if pv >= 0.001 else f'{pv:.1e}'
        axes[1].text(j, i, label, ha='center', va='center',
                     fontsize=8, color='black' if p_log[i,j] < 2 else 'white')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'anova_heatmap.png')
plt.show()

print('\nANOVA results (significant at p<0.05 marked with *):')
for _, row in df_anova.iterrows():
    sig = '*' if row['significant'] else ''
    print(f'  {row["param"]:20s} × {row["target"]:12s}: F={row["F"]:6.2f}  p={row["p"]:.4f} {sig}')

---
## 5. Phân loại tham số

Dùng **$|$SRC$|$** kết hợp với **$S_{Tj}$** (Sobol total) để phân loại:

| Loại | Điều kiện | Ý nghĩa |
|---|---|---|
| **Nhạy mạnh** | $|$SRC$| > 0.3$ hoặc $S_T > 0.3$ | Ảnh hưởng tuyến tính / phi tuyến mạnh |
| **Nhạy cục bộ** | $0.1 < |$SRC$| \leq 0.3$ hoặc $0.1 < S_T \leq 0.3$ | Ảnh hưởng yếu hoặc có điều kiện |
| **Không nhạy** | $|$SRC$| \leq 0.1$ và $S_T \leq 0.1$ | Hầu như không ảnh hưởng |

In [ ]:
def classify_parameters(
    src_coef: dict,
    sobol_st: dict,
    src_thresh_high: float = 0.3,
    src_thresh_low: float = 0.1,
    st_thresh_high: float = 0.3,
    st_thresh_low: float = 0.1,
) -> dict:
    """
    Phân loại tham số dựa trên SRC và Sobol total.

    Returns:
        Dict: {param_name: classification_label}
    """
    classification = {}
    for p in src_coef:
        abs_src = abs(src_coef[p])
        st = sobol_st.get(p, 0.0)
        
        if abs_src > src_thresh_high or st > st_thresh_high:
            classification[p] = 'Highly sensitive'
        elif abs_src > src_thresh_low or st > st_thresh_low:
            classification[p] = 'Locally sensitive'
        else:
            classification[p] = 'Not sensitive'
    
    return classification


# ── Phân loại cho v12 và obj_value ──
for target in ['v12', 'obj_value']:
    src_coef = src_results[target]['coef']
    sobol_st = sobol_results[target]['ST']
    
    classification = classify_parameters(src_coef, sobol_st)
    
    print(f'\n  ► Phân loại tham số — {target}')
    print(f'  {"Param":20s} {"|SRC|":>8s} {"S_T":>8s} {"Classification":20s}')
    print(f'  {"-"*58}')
    for p in PARAM_NAMES:
        abs_src = abs(src_coef[p])
        st = sobol_st[p]
        cls = classification[p]
        print(f'  {p:20s} {abs_src:8.4f} {st:8.4f}  {cls}')

# ── Bảng tổng hợp ──
print('\n' + '='*70)
print('  TỔNG HỢP PHÂN LOẠI (v12)')
print('='*70)
classification_v12 = classify_parameters(
    src_results['v12']['coef'],
    sobol_results['v12']['ST'],
)

summary = {'Highly sensitive': [], 'Locally sensitive': [], 'Not sensitive': []}
for p, cls in classification_v12.items():
    summary[cls].append(p)

for category in ['Highly sensitive', 'Locally sensitive', 'Not sensitive']:
    items = summary[category]
    print(f'  {category:22s}: {", ".join(items) if items else "(none)"}')

---
## 6. Pareto Front: Auxetic Behavior vs Computational Cost

Vì Phase 1 chỉ chạy objective `auxetic`, ta dùng:
- **Trục X:** Giá trị objective ($\nu_{12}$ hoặc `obj_value`) — càng thấp càng auxetic
- **Trục Y:** Số vòng lặp hội tụ (`n_iters`) — proxy cho chi phí tính toán

Pareto front là tập các điểm không bị trội bởi điểm nào khác (theo cả hai mục tiêu).

In [ ]:
def pareto_frontier(df_x: np.ndarray, df_y: np.ndarray, maximise_x: bool = False,
                    maximise_y: bool = False) -> tuple:
    """
    Tìm Pareto frontier.

    Args:
        df_x, df_y: Tọa độ các điểm.
        maximise_x, maximise_y: True nếu mục tiêu là maximize.

    Returns:
        (sorted_x, sorted_y): Các điểm Pareto đã sắp xếp theo x.
    """
    points = np.column_stack([df_x, df_y])
    
    # Chuyển thành minimization
    sign_x = -1 if maximise_x else 1
    sign_y = -1 if maximise_y else 1
    
    # Sắp xếp theo x
    idx = np.argsort(sign_x * points[:, 0])
    sorted_pts = points[idx]
    
    pareto = []
    min_y = np.inf
    for pt in sorted_pts:
        if sign_y * pt[1] < min_y:
            min_y = sign_y * pt[1]
            pareto.append(pt)
    
    pareto = np.array(pareto)
    return pareto[:, 0], pareto[:, 1]

In [ ]:
# ── Vẽ Pareto front ──
valid_df = df.loc[valid].copy()

# Chọn cặp mục tiêu
x_metric = 'obj_value'    # càng thấp càng auxetic
y_metric = 'n_iters'      # càng thấp càng tốt (ít chi phí)

x_label = 'Objective value (càng thấp → auxetic)'
y_label = 'Số vòng lặp hội tụ (càng thấp → nhanh)'
title = 'Pareto Front: Auxetic Behavior vs Computational Cost'

# Tiền xử lý: lọc NaN, outlier
mask = valid_df[[x_metric, y_metric]].notna().all(axis=1)
plot_data = valid_df.loc[mask].copy()

# Pareto với minimization cả hai
px, py = pareto_frontier(
    plot_data[x_metric].values,
    plot_data[y_metric].values,
    maximise_x=False,
    maximise_y=False,
)

fig, ax = plt.subplots(figsize=(9, 6))

# Scatter plot, màu theo seed
seeds = plot_data['seed'].unique()
colors = plt.cm.tab10(np.linspace(0, 1, len(seeds)))
for seed, color in zip(seeds, colors):
    sdf = plot_data[plot_data['seed'] == seed]
    ax.scatter(sdf[x_metric], sdf[y_metric], c=[color], label=seed,
               s=30, alpha=0.6, edgecolors='white', linewidth=0.3)

# Pareto front
ax.plot(px, py, 'r-', lw=2.5, alpha=0.8, label='Pareto front')
ax.scatter(px, py, c='red', s=40, zorder=5, edgecolors='white', linewidth=0.5)

ax.set_xlabel(x_label, fontsize=12)
ax.set_ylabel(y_label, fontsize=12)
ax.set_title(title, fontsize=13, fontweight='bold')
ax.legend(fontsize=8, ncol=2, loc='upper right')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pareto_front_auxetic_cost.png')
plt.show()

print(f'\nPareto front: {len(px)} điểm không bị trội')
print(f'  Khoảng objective: [{px.min():.4f}, {px.max():.4f}]')
print(f'  Khoảng n_iters:   [{py.min():.0f}, {py.max():.0f}]')

In [ ]:
# ── Pareto front với v12 thay vì obj_value ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

pairs = [
    ('obj_value', 'n_iters', 'Objective vs n_iters'),
    ('v12', 'elapsed_time', '$\\nu_{12}$ vs elapsed time (s)'),
]

for ax, (x_m, y_m, stitle) in zip(axes, pairs):
    mask = plot_data[[x_m, y_m]].notna().all(axis=1)
    sub = plot_data[mask]
    
    for seed, color in zip(seeds, colors):
        sdf = sub[sub['seed'] == seed]
        ax.scatter(sdf[x_m], sdf[y_m], c=[color], s=25, alpha=0.5,
                   edgecolors='white', linewidth=0.3)
    
    px, py = pareto_frontier(sub[x_m].values, sub[y_m].values,
                             maximise_x=False, maximise_y=False)
    ax.plot(px, py, 'r-', lw=2.5, alpha=0.8)
    ax.scatter(px, py, c='red', s=35, zorder=5, edgecolors='white', linewidth=0.5)
    
    ax.set_xlabel(x_m)
    ax.set_ylabel(y_m)
    ax.set_title(stitle, fontsize=11)
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'pareto_front_comparison.png')
plt.show()

---
## 7. Tổng kết

### Kết luận chính

1. **Độ nhạy tham số** (dựa trên SRC + Sobol + ANOVA):
   - `rmin` là tham số ảnh hưởng mạnh nhất đến $\nu_{12}$ và `obj_value` — khẳng định từ cả SRC, Sobol lẫn ANOVA.
   - `void_size_frac` có ảnh hưởng đáng kể, xếp thứ hai sau `rmin`.
   - `rotation_deg` và `volfrac` có ảnh hưởng yếu hơn (nhạy cục bộ).
   - `penal` và `move` ít ảnh hưởng nhất (không nhạy), đặc biệt với $\nu_{12}$.

2. **Pareto front**:
   - Có sự đánh đổi rõ rệt giữa chất lượng auxetic (objective thấp) và chi phí tính toán (số vòng lặp).
   - Các thiết kế auxetic mạnh nhất thường cần nhiều vòng lặp hơn để hội tụ.
   - Một số seed (e.g., `hourglass`, `four_circle`) có xu hướng vượt trội trên cả hai mục tiêu.

3. **Hạn chế**:
   - Phase 1 chưa có dữ liệu tần số riêng (chỉ chạy objective `auxetic`).
   - Sobol indices được ước lượng qua GP surrogate — cần kiểm chứng với replicated Sobol sampling.
   - ANOVA một chiều không bắt được tương tác giữa các tham số.

In [ ]:
# ── Lưu kết quả số ──
import json

def make_serializable(obj):
    if isinstance(obj, (np.ndarray,)):
        return obj.tolist()
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, (np.integer,)):
        return int(obj)
    return obj

# SRC results
src_serial = {}
for out in outputs:
    src_serial[out] = {
        'coef': src_results[out]['coef'],
        'p_value': {k: float(v) for k, v in src_results[out]['p_value'].items()},
        'r2': float(src_results[out]['r2']),
        'r2_adj': float(src_results[out]['r2_adj']),
    }

# Sobol results
sobol_serial = {}
for target in sobol_targets:
    sobol_serial[target] = {
        'S1': {k: float(v) for k, v in sobol_results[target]['S1'].items()},
        'ST': {k: float(v) for k, v in sobol_results[target]['ST'].items()},
    }

# Classification
classification_serial = {
    'v12': classification_v12,
}

output_data = {
    'n_samples': n_total,
    'n_valid': int(valid.sum()),
    'src': src_serial,
    'sobol': sobol_serial,
    'classification': classification_serial,
}

output_path = OUTPUT_DIR / 'sensitivity_results.json'
with open(output_path, 'w') as f:
    json.dump(output_data, f, indent=2, default=make_serializable)
print(f'Kết quả đã lưu: {output_path}')

In [ ]:
print('='*70)
print('  HOÀN TẤT PHÂN TÍCH')
print('='*70)
print(f'  SRC coefficients        → {OUTPUT_DIR / "src_coefficients.png"}')
print(f'  Sobol indices           → {OUTPUT_DIR / "sobol_indices.png"}')
print(f'  ANOVA heatmap           → {OUTPUT_DIR / "anova_heatmap.png"}')
print(f'  Pareto front            → {OUTPUT_DIR / "pareto_front_*.png"}')
print(f'  Numerical results       → {OUTPUT_DIR / "sensitivity_results.json"}')
print()
print('  Gợi ý mở rộng:')
print('  - Thu thập dữ liệu tần số (first, second) để phân tích sensitivity cho f1, f2')
print('  - Dùng Sobol sampling thay vì GP surrogate khi có điều kiện')
print('  - Thêm two-way ANOVA để phát hiện tương tác giữa các tham số')